**`ingest_transactions`**

Ingest deed / real-estate transaction records into openplaces.

Examples:

- `US-WI_transaction-widor-2026`.

  Wisconsin Statewide Real Estate Transfer Return (RETR) data.

  A browser scraper downloads one CSV per year-month from the WI DOR portal.
  
- `US-MA_transaction-masslandrecords-v1`

  Massachusetts registry.

  Deed records crawled from Avenu WebForms (masslandrecords.com).

# Configure

In [ ]:
import argparse

import openplaces as op
from openplaces.recipe import get_recipe_by_id
from openplaces.utils import pretty_print

In [ ]:
# Define arguments (shared by both sources)
parser = argparse.ArgumentParser(
    description='Ingest transaction records using a recipe'
)
parser.add_argument(
    '--recipe_id',
    help='Recipe ID, e.g. "US-WI_transaction-widor-2026"',
)
parser.add_argument(
    '--admin_ids',
    help='Admin IDs, e.g. "US-WI" or "US-MA-MI-SO"',
    nargs='*',
)
parser.add_argument(
    '--partition_ids',
    help='Specific year-month partitions, e.g. "202601". '
    'Omit for the full recipe range.',
    nargs='*',
)
parser.add_argument(
    '--years',
    type=int,
    nargs='*',
    help='Four-digit calendar years, e.g. "2024". Omit for the full recipe range.',
)
parser.add_argument(
    '--reprocess',
    help='Reprocess already-ingested partitions',
    action='store_true',
)
parser.add_argument(
    '--redownload',
    help='Re-download source files (WI)',
    action='store_true',
)
parser.add_argument('--verbose', action='store_true')

# Test arguments

In [ ]:
ARGS_TEST = (
    # Wisconsin RETR (browser-downloaded monthly CSVs; working)
    '--recipe_id US-WI_transaction-widor-2026 '
    '--partition_ids 202601 '  # a specific year-month
    # '--partition_ids 202512 202601 202602 '  # several months; omit for full range
    #
    # Massachusetts registry (Avenu crawler; in progress)
    # '--recipe_id US-MA_transaction-masslandrecords-v1 '
    # '--admin_ids US-MA-MI-CA '  # Cambridge
    # '--admin_ids US-MA-MI-SO '  # Somerville
    # '--admin_ids US-MA-MI-ME '  # Medford
    # '--years 2026 2025 2024 2023 2022 '  # whole years; omit to crawl the full available range
    # '--partition_ids 202601 '  # a specific MA registry month
    #
    # Transactions in New Hanover, North Carolina
    # '--recipe_id US-NC-NE_transaction-nhcgov-2026 '
    # '--admin_ids US-NC-NE '
    # '--reprocess '
    # '--redownload '
    '--verbose '
)

args_list = [x for x in ARGS_TEST.split(' ') if x]
args = parser.parse_args(args_list)
args

In [ ]:
# Show recipe parameters
pretty_print(get_recipe_by_id(args.recipe_id))

# Ingest data

In [ ]:
op.ingest(
    args.recipe_id,
    admin_ids=args.admin_ids,
    partition_ids=args.partition_ids,
    years=args.years,
    reprocess=args.reprocess,
    redownload=args.redownload,
    verbose=args.verbose,
)

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script, test_script

COMMIT = True
# If True, writes `.py` scripts to 'scripts/'.
# If False, writes a test version of the script to 'scripts/_test/'

In [ ]:
convert_to_script(commit=COMMIT)

# Test script

In [ ]:
test_script(*args_list, committed=COMMIT)

# Inspect output

In [ ]:
transactions = op.get_entities(
    args.recipe_id,
    admin_id=args.admin_ids,
    missing='warn',
)
op.inspect_table(transactions)

In [ ]:
import numpy as np

from openplaces.viz import add_log_ticks

mask = transactions['price'].gt(0)
ax = transactions[mask]['price'].apply(np.arcsinh).hist(bins=100, figsize=(10, 2))
add_log_ticks(ax, prefix='$')
ax.set_xlabel('price')
ax.set_title(
    f'Sale price distribution: {args.admin_ids[0] if args.admin_ids else args.recipe_id}'
)